# Experiment: Case-001 Branch Review Handoff Status Gate

Objective: verify that public packet-checker output plus optional public response-capture output collapses into one handoff label without private fields or patient-specific action.

Success criteria:
- packet incomplete stays blocked;
- packet ready without response asks only the private clinician question;
- response follow-up, release-blocked, and captured states map to distinct public labels;
- public output never contains private-only keys.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = next(
        parent for parent in Path.cwd().parents if (parent / "scripts").exists()
    )

sys.path.insert(0, str(REPO_ROOT))

from scripts.summarize_branch_review_handoff import (  # noqa: E402
    HANDOFF_NEEDS_FOLLOWUP_LABEL,
    HANDOFF_PACKET_INCOMPLETE_LABEL,
    HANDOFF_READY_FOR_QUESTION_LABEL,
    HANDOFF_RELEASE_BLOCKED_LABEL,
    HANDOFF_RESPONSE_CAPTURED_LABEL,
    PRIVATE_ONLY_KEYS,
    summarize_handoff,
)

## Synthetic Inputs

Inputs mimic public JSON emitted by `check_branch_review_packet.py` and `capture_branch_review_response.py`. They intentionally contain no raw record, doctor message, contact, identity, treatment instruction, or sample route.

In [ ]:
packet_incomplete = {
    "current_label": "branch_review_packet_not_ready_public_only",
    "ready_domains": ["diagnosis_genotype_phenotype"],
    "missing_domains": [
        "transfusion_burden_response",
        "blood_bank_immune_history",
        "iron_organ_chelation_status",
        "hct_gene_cell_access_context",
        "consent_ethics_private_owner_review",
    ],
    "public_labels": [],
    "blocked_outputs": [],
}

packet_ready = {
    "current_label": "branch_review_packet_ready_for_private_clinician_review",
    "ready_domains": [
        "diagnosis_genotype_phenotype",
        "transfusion_burden_response",
        "blood_bank_immune_history",
        "iron_organ_chelation_status",
        "hct_gene_cell_access_context",
        "consent_ethics_private_owner_review",
    ],
    "missing_domains": [],
    "public_labels": [],
    "blocked_outputs": [],
}

response_followup = {
    "current_label": "branch_review_response_needs_private_followup",
    "captured_labels": ["cannot_answer_from_packet"],
    "missing_domains": ["iron_organ_chelation_status"],
    "branch_scope_labels": [],
    "blocked_outputs": [],
}

response_blocked = {
    "current_label": "branch_review_response_release_blocked_private_material",
    "captured_labels": ["release_blocked_private_material"],
    "missing_domains": [],
    "branch_scope_labels": [],
    "blocked_outputs": [],
}

response_captured = {
    "current_label": "branch_review_response_captured_public_labels_only",
    "captured_labels": ["branch_in_scope_for_private_review_allogeneic_hct"],
    "missing_domains": [],
    "branch_scope_labels": ["branch_in_scope_for_private_review_allogeneic_hct"],
    "blocked_outputs": [],
}

## Handoff State Checks

The only state transition this notebook tests is public workflow status. It does not decide whether any therapy, trial, referral, transfusion change, chelation change, lab contact, or sample movement is appropriate.

In [ ]:
scenarios = {
    "packet_incomplete": (
        summarize_handoff(packet_incomplete),
        HANDOFF_PACKET_INCOMPLETE_LABEL,
    ),
    "packet_ready_no_response": (
        summarize_handoff(packet_ready),
        HANDOFF_READY_FOR_QUESTION_LABEL,
    ),
    "response_followup": (
        summarize_handoff(packet_ready, response_followup),
        HANDOFF_NEEDS_FOLLOWUP_LABEL,
    ),
    "response_blocked": (
        summarize_handoff(packet_ready, response_blocked),
        HANDOFF_RELEASE_BLOCKED_LABEL,
    ),
    "response_captured": (
        summarize_handoff(packet_ready, response_captured),
        HANDOFF_RESPONSE_CAPTURED_LABEL,
    ),
}

for name, (summary, expected_label) in scenarios.items():
    assert summary["current_label"] == expected_label, name
    assert not PRIVATE_ONLY_KEYS.intersection(summary), name

{name: summary["current_label"] for name, (summary, _) in scenarios.items()}

## Result

The handoff gate is useful because it gives the founder one public-safe state to say to a clinician or collaborator: packet incomplete, ready for private clinician question, needs follow-up, release blocked, or response captured. It does not turn branch labels into clinical action.